In [15]:
"""
Figure X - kynurenine and physical resilience

Panels:
  - forest plot of within-person age slope differences across the tested panel
  - scatter of the focus compound against chronological age
  - per-participant slopes for the focus compound, compared between groups

Age enters the model as a within-participant term: visit age minus that
participant's mean age. The effect tested is therefore change within people as
they age, not differences between older and younger participants. Figures plot
chronological age, which reads more naturally, with an ordinary least squares
trend per group for orientation.

The per-participant slope figure is an independent check on the mixed model:
each participant's own slope is fitted from their three visits, then the two
groups are compared with a rank test that makes no distributional assumption.
"""

import os, re, warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests

warnings.filterwarnings("ignore")
matplotlib.rcParams["svg.fonttype"] = "none"

BASE_DIR = r"C:\Users\sirju\OneDrive\Desktop\AgingPaper2\Tryptophan_metabolites"
OUT_DIR  = os.path.join(BASE_DIR, "figures")

ALL_VISITS = "0-other-res-df-all-visits.csv"
GROUPS     = "0-other-res-df-group.csv"
GNPS       = "885c8595f0574824a2bdef9d6ee2db88-merged_results_with_gnps.tsv"

ID, AGE, GROUP = "record_ID", "host_age", "physresilience_group"
MIN_MQSCORE, MIN_DETECTION = 0.70, 0.70
FOCUS = "Kynurenine"          # compound shown in panels b and c

PATHWAY = (r"tryptophan|tryptamin|kynuren|quinolinic|xanthurenic|picolinic|anthranil|"
           r"indol|indox|serotonin|melatonin|hydroxyindole|nicotinamide|niacin|"
           r"quinaldic|skatole")

COMPOUNDS = [
    ("L-Tryptophan",                    r"^l-tryptophan"),
    ("Kynurenine",                      r"kynurenine"),
    ("Indole-3-lactate",                r"indole-?3-?lactic|indolelactic"),
    ("Glucopyranosyl-L-tryptophan",     r"glucopyranosyl.*tryptophan"),
    ("5-Methylindole-3-carboxaldehyde", r"methylindole-?3-?carboxaldehyde"),
]

BLUE, RED = "#2567B5", "#B87400"   # high resilience (blue), low resilience (amber)

LEGEND = {"High": "High physical resilience",
          "Low":  "Low physical resilience"}


def feature_map(df):
    skip = {ID, AGE, GROUP, "visit", "filename"}
    out = {}
    for c in df.columns:
        if c in skip:
            continue
        scan = c[1:] if c.startswith("X") and c[1:].isdigit() else c
        if scan.isdigit():
            out[scan] = c
    return out


def load():
    d = pd.read_csv(os.path.join(BASE_DIR, ALL_VISITS), low_memory=False)
    scan2col = feature_map(d)
    ref = pd.read_csv(os.path.join(BASE_DIR, GROUPS), low_memory=False)
    d[GROUP] = d[ID].map(ref.set_index(ID)[GROUP])
    d = d[d[GROUP].notna()].copy()
    if (d.groupby(ID)[GROUP].nunique() > 1).any():
        raise ValueError("group label is not constant within a participant")
    d["age_within"] = d[AGE] - d.groupby(ID)[AGE].transform("mean")
    print(f"{len(d)} visits, {d[ID].nunique()} participants")
    print(f"groups: {d.drop_duplicates(ID)[GROUP].value_counts().to_dict()}")
    return d, scan2col


def rclr(matrix):
    m = np.asarray(matrix, dtype=float)
    logm = np.where(m > 0, np.log(np.where(m > 0, m, 1.0)), np.nan)
    return logm - np.nanmean(logm, axis=1, keepdims=True)


def build_panel(d, scan2col):
    lib = pd.read_csv(os.path.join(BASE_DIR, GNPS), sep="\t", low_memory=False)
    lib = lib.sort_values("MQScore", ascending=False).drop_duplicates("#Scan#")
    lib = lib[lib.MQScore >= MIN_MQSCORE].copy()
    lib["scan"] = lib["#Scan#"].astype(str)
    hits = lib[lib.Compound_Name.astype(str).str.lower()
                  .str.contains(PATHWAY, regex=True, na=False)
               & lib.scan.isin(scan2col)].copy()
    hits["column"]    = hits.scan.map(scan2col)
    hits["detection"] = [(d[c] > 0).mean() for c in hits.column]
    panel = hits[hits.detection >= MIN_DETECTION].copy()

    def name_of(s):
        s = str(s).lower()
        for label, pattern in COMPOUNDS:
            if re.search(pattern, s):
                return label
        return str(s)[:40]

    panel["compound"] = panel.Compound_Name.map(name_of)
    panel = panel.sort_values(["compound", "detection", "MQScore"],
                              ascending=[True, False, False])
    panel = panel.drop_duplicates("compound", keep="first")
    print(f"\ncompounds tested: {len(panel)}")
    for _, r in panel.iterrows():
        print(f"  {r.compound:<34} scan {r.scan:<7} {r.detection:>4.0%}")
    return panel


def test_panel(d, R, col_index, panel):
    rows, frames = [], {}
    for _, meta in panel.iterrows():
        df = pd.DataFrame({
            "y":       R[:, col_index[meta.column]],
            "age":     d[AGE],                # chronological age, for plotting
            "within":  d["age_within"],       # within-person age, for the model
            "group":   d[GROUP],
            "low":     (d[GROUP] == "Low").astype(int),
            "id":      d[ID],
        }).dropna()
        m = smf.mixedlm("y ~ within * low", df, groups=df["id"]).fit(reml=False)
        frames[meta.compound] = (m, df)
        rows.append({
            "compound":    meta.compound,
            "scan":        meta.scan,
            "detection_%": round(meta.detection * 100),
            "n_visits":    len(df),
            "slope_High":  m.params["within"],
            "slope_Low":   m.params["within"] + m.params["within:low"],
            "slope_diff":  m.params["within:low"],
            "slope_SE":    m.bse["within:low"],
            "p":           m.pvalues["within:low"],
            "singular":    bool(m.cov_re.iloc[0, 0] < 1e-8),
        })
    res = pd.DataFrame(rows)
    res["FDR"] = multipletests(res.p, method="fdr_bh")[1]
    return res.sort_values("p").reset_index(drop=True), frames


def participant_slopes(df):
    """Fit each participant's own slope across their visits."""
    out = []
    for pid, s in df.groupby("id"):
        if len(s) >= 3 and s.within.std() > 0:
            out.append((pid, s.group.iloc[0], np.polyfit(s.within, s.y, 1)[0]))
    return pd.DataFrame(out, columns=["id", "group", "slope"])


def save(fig, path):
    fig.tight_layout()
    fig.savefig(path, format="svg", bbox_inches="tight")
    plt.close(fig)
    print(f"wrote {path}")


def plot_forest(res, path):
    """Difference in within-person age slope for every compound tested."""
    fig, ax = plt.subplots(figsize=(6.4, 0.6 * len(res) + 2.0))
    ordered = res.sort_values("slope_diff")

    for i, (_, r) in enumerate(ordered.iterrows()):
        sig = r.FDR < 0.05
        colour = "#2B2B2B" if sig else "#B4BAC2"
        ax.errorbar(r.slope_diff, i, xerr=1.96 * r.slope_SE, fmt="o",
                    color=colour, ms=7 if sig else 6, capsize=3,
                    lw=2.0 if sig else 1.4)
        if sig:
            ax.text(r.slope_diff + 1.96 * r.slope_SE + 0.0015, i,
                    f"FDR = {r.FDR:.3f}", fontsize=9, ha="left",
                    va="center", color="#2B2B2B")

    ax.axvline(0, ls="--", c="k", lw=1)
    ax.set_yticks(range(len(ordered)))
    ax.set_yticklabels(ordered.compound, fontsize=10.5)
    ax.set_xlabel("Δ age slope (low − high)", fontsize=13)
    ax.set_title("Tryptophan-pathway panel", fontsize=11.5, loc="left")
    ax.spines[["top", "right"]].set_visible(False)
    ax.tick_params(labelsize=10)
    save(fig, path)


def plot_scatter(row, df, path):
    """Metabolite against chronological age, one point per visit.

    The trend lines are ordinary least squares per group, shown for orientation
    only; the statistics quoted come from the mixed model on within-person age.
    Legend counts are the participants actually plotted, which is everyone with
    at least one visit where the compound was detected.
    """
    fig, ax = plt.subplots(figsize=(5.6, 4.8))

    for label, colour in [("High", BLUE), ("Low", RED)]:
        s = df[df.group == label]
        ax.scatter(s.age, s.y, s=10, color=colour, alpha=0.28, lw=0, zorder=2)

    for label, colour in [("High", BLUE), ("Low", RED)]:
        s = df[df.group == label]
        grid = np.linspace(s.age.min(), s.age.max(), 120)
        X = np.column_stack([np.ones(len(s)), s.age.values])
        beta, *_ = np.linalg.lstsq(X, s.y.values, rcond=None)
        resid = s.y.values - X @ beta
        sigma2 = resid @ resid / (len(s) - 2)
        XtX_inv = np.linalg.inv(X.T @ X)
        G = np.column_stack([np.ones_like(grid), grid])
        fit = G @ beta
        se = np.sqrt(sigma2 * np.einsum("ij,jk,ik->i", G, XtX_inv, G))
        ax.fill_between(grid, fit - 1.96 * se, fit + 1.96 * se,
                        color=colour, alpha=0.30, lw=0, zorder=8)
        ax.plot(grid, fit, color=colour, lw=3.4, zorder=9,
                label=f"{LEGEND[label]} (n = {s['id'].nunique()})")

    ax.set_xlabel("Chronological age (years)", fontsize=13)
    ax.set_ylabel(f"{row.compound} (rclr)", fontsize=13)
    ax.set_title(f"{row.compound}\nFDR = {row.FDR:.3f}", fontsize=11.5, loc="left")
    ax.spines[["top", "right"]].set_visible(False)
    ax.tick_params(labelsize=10)
    ax.legend(fontsize=10, frameon=False)
    save(fig, path)


def plot_slopes(high, low, p_rank, compound, path):
    """Each participant's own slope, compared between groups."""
    fig, ax = plt.subplots(figsize=(4.4, 4.8))
    rng = np.random.default_rng(0)

    for i, (values, colour) in enumerate([(high, BLUE), (low, RED)]):
        ax.scatter(rng.normal(i + 1, 0.075, len(values)), values,
                   s=15, color=colour, alpha=0.55, lw=0, zorder=2)

    ax.boxplot([high, low], widths=0.55, showfliers=False, zorder=3,
               patch_artist=True,
               boxprops=dict(facecolor="none", edgecolor="#2B2B2B", lw=1.2),
               medianprops=dict(color="#2B2B2B", lw=1.8),
               whiskerprops=dict(color="#2B2B2B", lw=1.2),
               capprops=dict(color="#2B2B2B", lw=1.2))

    ax.axhline(0, ls=":", c="grey", lw=1)
    ax.set_xticks([1, 2])
    ax.set_xticklabels([f"High\n(n={len(high)})", f"Low\n(n={len(low)})"], fontsize=11)
    ax.set_ylabel("Individual slope", fontsize=13)
    ax.set_title(f"{compound}\nMann-Whitney P = {p_rank:.4f}", fontsize=11.5, loc="left")
    ax.spines[["top", "right"]].set_visible(False)
    ax.tick_params(labelsize=10)
    save(fig, path)


def main():
    os.makedirs(OUT_DIR, exist_ok=True)
    d, scan2col = load()
    columns = list(scan2col.values())
    R = rclr(d[columns].to_numpy(dtype=float))
    col_index = {c: i for i, c in enumerate(columns)}
    panel = build_panel(d, scan2col)
    res, frames = test_panel(d, R, col_index, panel)

    pd.set_option("display.width", 200)
    print("\nresults")
    print(res.round(5).to_string(index=False))

    table = os.path.join(OUT_DIR, "tryptophan_results.csv")
    res.to_csv(table, index=False)
    print(f"\nwrote {table}")

    plot_forest(res, os.path.join(OUT_DIR, "forest_age_slope.svg"))

    row = res[res.compound == FOCUS].iloc[0]
    model, df = frames[FOCUS]
    slopes = participant_slopes(df)
    high = slopes.slope[slopes.group == "High"]
    low  = slopes.slope[slopes.group == "Low"]
    p_rank = mannwhitneyu(high, low)[1]

    safe = re.sub(r"[^A-Za-z0-9]+", "_", FOCUS).strip("_")
    plot_scatter(row, df, os.path.join(OUT_DIR, f"{safe}_by_age.svg"))
    plot_slopes(high, low, p_rank, FOCUS,
                os.path.join(OUT_DIR, f"{safe}_participant_slopes.svg"))
    print(f"\nscatter: {df['id'].nunique()} participants, {len(df)} visits")
    print(f"slope figure: {len(slopes)} participants with the compound "
          f"detected at all three visits")
    print(f"per-participant slope test: High median {high.median():.4f}, "
          f"Low median {low.median():.4f}, Mann-Whitney P = {p_rank:.4f}")


if __name__ == "__main__":
    main()












    """Checks that the rclr transform in FigureX_kynurenine.py behaves as intended.

Four things must hold if the transform is row-wise (per sample):
  1. every sample's rclr values average to zero
  2. within a sample, log(raw) - rclr is the same constant for every feature
  3. that constant equals the mean log of that sample's detected features
  4. positions that were zero in the input are missing in the output
"""

import numpy as np
import pandas as pd

BASE_DIR   = r"C:\Users\sirju\OneDrive\Desktop\AgingPaper2\Tryptophan_metabolites"
ALL_VISITS = "0-other-res-df-all-visits.csv"
ID, AGE = "record_ID", "host_age"


def rclr(matrix):
    m = np.asarray(matrix, dtype=float)
    logm = np.where(m > 0, np.log(np.where(m > 0, m, 1.0)), np.nan)
    return logm - np.nanmean(logm, axis=1, keepdims=True)


import os
d = pd.read_csv(os.path.join(BASE_DIR, ALL_VISITS), low_memory=False)
skip = {ID, AGE, "visit", "filename"}
cols = [c for c in d.columns
        if c not in skip and str(c).lstrip("X").isdigit()]
M = d[cols].to_numpy(dtype=float)
R = rclr(M)

print(f"matrix: {M.shape[0]} samples x {M.shape[1]} features")
print(f"zeros in input: {(M == 0).sum():,}\n")

# 1. each sample centres on zero
row_means = np.nanmean(R, axis=1)
print(f"1. per-sample mean of rclr   max |mean| = {np.nanmax(np.abs(row_means)):.2e}"
      f"   {'PASS' if np.nanmax(np.abs(row_means)) < 1e-9 else 'FAIL'}")

# 2. the value subtracted is constant across features within a sample
logM = np.where(M > 0, np.log(np.where(M > 0, M, 1.0)), np.nan)
implied = logM - R                       # what was subtracted, per cell
spread = np.nanmax(implied, axis=1) - np.nanmin(implied, axis=1)
print(f"2. spread of subtracted value within a sample   max = {np.nanmax(spread):.2e}"
      f"   {'PASS (row-wise)' if np.nanmax(spread) < 1e-9 else 'FAIL (not row-wise)'}")

# 3. that constant is the sample's own mean log of detected features
centre_expected = np.nanmean(logM, axis=1)
centre_actual   = np.nanmean(implied, axis=1)
print(f"3. subtracted value equals sample mean log      max diff = "
      f"{np.nanmax(np.abs(centre_expected - centre_actual)):.2e}"
      f"   {'PASS' if np.nanmax(np.abs(centre_expected - centre_actual)) < 1e-9 else 'FAIL'}")

# 4. zeros stay missing, non-zeros stay present
ok_zero    = np.isnan(R[M == 0]).all()
ok_nonzero = np.isfinite(R[M > 0]).all()
print(f"4. zeros -> NaN, detected -> finite            "
      f"{'PASS' if ok_zero and ok_nonzero else 'FAIL'}")

# worked example for one sample
i = 0
nz = M[i] > 0
print(f"\nworked example - sample {d[ID][i]} / {d['visit'][i]}")
print(f"  detected features        : {nz.sum():,} of {len(nz):,}")
print(f"  centre (mean log)        : {np.log(M[i][nz]).mean():.5f}")
for scan in ["8541", "13164", "13152"]:
    if scan in cols:
        j = cols.index(scan)
        if M[i, j] > 0:
            print(f"  scan {scan:<6} raw {M[i, j]:>12,.1f}"
                  f"  log {np.log(M[i, j]):8.4f}"
                  f"  rclr {R[i, j]:8.4f}")

711 visits, 237 participants
groups: {'Low': 119, 'High': 118}

compounds tested: 5
  5-Methylindole-3-carboxaldehyde    scan 8058     80%
  Glucopyranosyl-L-tryptophan        scan 12022    90%
  Indole-3-lactate                   scan 13164   100%
  Kynurenine                         scan 8541     98%
  L-Tryptophan                       scan 13152   100%

results
                       compound  scan  detection_%  n_visits  slope_High  slope_Low  slope_diff  slope_SE       p  singular     FDR
                     Kynurenine  8541           98       700     0.02399    0.03892     0.01493   0.00561 0.00783     False 0.03913
               Indole-3-lactate 13164          100       710     0.01855    0.03218     0.01362   0.00634 0.03173     False 0.05899
                   L-Tryptophan 13152          100       710     0.01738    0.03035     0.01297   0.00616 0.03539     False 0.05899
    Glucopyranosyl-L-tryptophan 12022           90       642     0.01554    0.04148     0.02594   0.0137